### 📦 Offline Dependency Installation

- Installs `datasets` and `trl` into a local directory (`/kaggle/working/packages`)
- Prefers **offline packages** (for reproducibility & no internet dependency)
- Falls back to PyPI if offline packages are unavailable
- Appends install path to `sys.path`

✅ Ensures Kaggle-compatible, reproducible environment  
⚠️ Avoids permission issues with global installs

In [1]:
# ============================================================
# Cell 1: 环境安装与路径设置
# 作用：在Kaggle离线环境中安装所需Python包
# ============================================================

import subprocess# 用来在Python内部执行系统命令（相当于在终端输入命令）
import sys# 访问Python解释器相关的变量和函数
import os          # 操作文件和文件夹

from pathlib import Path  # 更现代的文件路径处理库

def resolve_python_path(target_dir):
    """
    扫描目标目录下所有的.pth文件，把里面记录的路径加入Python搜索路径。
    .pth文件是Python的路径配置文件，每行写一个额外的包路径。
    这样Python就能import那个路径下的包。
    """
    for pth_file in Path(target_dir).glob("*.pth"):
        # glob("*.pth")：匹配该目录下所有以.pth结尾的文件
        with pth_file.open() as fp:
            relpath = fp.read().strip()  # 读取文件内容，strip()去掉首尾空白/换行
            rel_pack_path = pth_file.parent / relpath
            # 用.pth文件所在的目录，拼接读取到的相对路径，得到绝对路径
            if rel_pack_path.exists():
                print(f"append {rel_pack_path}")
                sys.path.append(str(rel_pack_path))
                # 把这个路径加入sys.path，Python就能从这里import包

# ---- 定义关键目录 ----
offline_dir = "/kaggle/input/nvidia-nemotron-offline-packages/offline_packages"
# Kaggle预先下载好的离线包目录（Kaggle训练环境没有网络）

target_dir = "/kaggle/working/packages"
# 我们把包安装到这个目录

os.makedirs(target_dir, exist_ok=True)
# 创建目标目录，exist_ok=True表示目录已存在时不报错

# 把NVIDIA工具脚本的路径加入Python搜索路径
resolve_python_path("/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/")

# ---- 从离线目录安装包 ----
if os.path.exists(offline_dir):
    try:
        subprocess.check_call([
            sys.executable,# 当前Python解释器路径
            "-m", "pip",          # 用pip
            "install",            # 安装
            "-q",                 # 安静模式，减少输出
            "--no-index",         # 不去PyPI网络下载，只用本地文件
            "--find-links", offline_dir,# 在offline_dir里寻找安装包
            "--target", target_dir,# 安装到target_dir（不是全局）"datasets",           # HuggingFace数据集处理库
            "trl"                 # Transformer强化学习库（含SFTTrainer）
        ])
        print("✅ 从离线包安装成功")
    except subprocess.CalledProcessError as e:
        # 如果安装失败，打印错误信息，尝试在线安装作为备用
        print(f"⚠️ 离线安装失败: {e}，尝试在线安装...")
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q",
            "datasets", "trl"
        ])
else:
    # 如果没有离线包目录，直接在线安装
    print("未找到离线包目录，尝试在线安装...")
    try:
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q",
            "datasets", "trl"
        ])
        print("✅ 在线安装成功")
    except subprocess.CalledProcessError as e:
        print(f"❌ 安装失败: {e}")

# ---- 把安装好的包目录加入Python搜索路径 ----
sys.path.append(target_dir)
resolve_python_path(target_dir)

# ---- 测试导入 ----
try:
    import datasets
    import cutlass
    print(f"✅ datasets版本: {datasets.__version__}")
    print("✅ cutlass导入成功")
except ImportError as e:
    print(f"⚠️ 导入警告: {e}，部分功能可能受限")

append /kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/nvidia_cutlass_dsl/python_packages


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.31.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.21.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
torchvision 0.26.0.dev20260324+cu128 requires torch==2.12.0.dev20260324, but you have torch 2.11.0 which is incompatible.
cuda-python 12.9.6 requires cuda-bindings~=12.9.6, but you have cuda-bindings 13.2.0 which is incompatible.
ydata-profiling 4.18.1 requires numpy<2.4,>=1.22, but you have numpy 2.4.4 which is incompatible.
ydata-profiling 4.18.1 requires pandas!=1.4.0,<3.0,>1.5, but you have pandas 3.0.1 which is incompatible.
ydata-profiling 4.18.1 requires scipy<1.17,>=1.8, but you have scipy 1.17.1 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatibl

✅ 从离线包安装成功
✅ datasets版本: 4.0.0
✅ cutlass导入成功


### ⚙️ Environment Setup & Libraries

- Enables `expandable_segments` → reduces CUDA memory fragmentation (important for 30B)
- Imports:
  - `torch`, `transformers` → model training
  - `polars` → fast data loading
  - `datasets` → HF dataset wrapper
  - `trl`, `peft` → SFT + LoRA fine-tuning
  - `kagglehub` → model download

✅ Optimized for large-model training on H100

In [2]:
# ============================================================
# Cell 2: 核心库导入（修复kagglehub卡住问题）
# 必须在import torch之前设置CUDA内存配置！
# ============================================================

import os

# ⭐ 关键：必须在import torch之前设置，否则不生效
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
# 作用：让GPU内存可以动态扩展分配，大幅减少"CUDA out of memory"错误

#⭐ 修复kagglehub卡住问题：在import之前设置这些环境变量
os.environ["KAGGLE_DATA_PROXY_PROJECT"] = "kaggle-161607"
# 告诉kagglehub使用Kaggle内部代理，而不是去连接外网
# 这是Kaggle notebook内置的数据代理服务地址

os.environ["KAGGLE_DATA_PROXY_URL"] = "https://dp.kaggle.net"
# Kaggle数据代理的URL，kagglehub通过这个URL下载模型/数据集
# 在Kaggle环境里这个地址是可达的（不需要外网）

os.environ["KAGGLE_URL_BASE"] = "https://www.kaggle.com"
# Kaggle主站地址，某些API请求会用到

# ⭐ 额外保险：设置超时，防止网络请求无限等待
os.environ["KAGGLE_DATA_PROXY_TOKEN"] = os.environ.get(
    "KAGGLE_DATA_PROXY_TOKEN", ""
)
# 读取Kaggle环境中已有的token（Kaggle notebook会自动注入这个变量）
# 如果不存在就用空字符串，避免KeyError

import sys
import stat
import shutil
import gc
import zipfile

import polars as pl

import torch
import torch.nn.functional as F

# ---- 现在再import kagglehub（已经设置好环境变量，不会卡住了）----
import kagglehub

from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig

# ---- 打印GPU信息，确认环境正常 ----
print(f"✅ PyTorch版本: {torch.__version__}")
print(f"✅ CUDA是否可用: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU型号: {torch.cuda.get_device_name(0)}")
    total_mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"✅ GPU显存总量: {total_mem:.1f} GB")
else:
    print("❌ 未检测到GPU，请在Kaggle中开启GPU加速器！")
    print("   Settings → Accelerator → 选择GPU T4/P100")

/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/torch/compiler/__init__.py:148: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  return torch._dynamo.allow_in_graph(fn)


✅ PyTorch版本: 2.12.0.dev20260324+cu128
✅ CUDA是否可用: True
✅ GPU型号: NVIDIA RTX PRO 6000 Blackwell Server Edition
✅ GPU显存总量: 95.0 GB


### 🛠️ Triton + RMSNorm Fix (Kaggle Hack)

- Replaces `rmsnorm_fn` with a pure PyTorch implementation
  → avoids Triton kernel crashes

- Copies & patches `ptxas` binary for Blackwell compatibility
- Overrides Triton backend paths + permissions

✅ Fixes incompatibilities between:
  - Kaggle runtime
  - Triton compiler
  - Nemotron kernels

⚠️ Hacky but necessary for stable training

In [3]:
# ============================================================
# Cell 3: Triton兼容性修复
# 背景：Triton是NVIDIA的GPU编程语言，在Kaggle Blackwell GPU上有兼容性问题
# 解决方案：
#   1. 用纯PyTorch实现替换有问题的Triton版RMSNorm
#   2. 复制并修复ptxas（GPU汇编编译器）的路径和权限
# ============================================================

# ---- 修复1：用纯PyTorch实现RMSNorm ----
def _pure_rmsnorm_fn(x, weight, bias=None, z=None, eps=1e-5,
                     group_size=None, norm_before_gate=True, upcast=True):
    """
    RMSNorm（Root Mean Square Normalization，均方根归一化）的纯PyTorch实现。
    
    RMSNorm是LayerNorm的简化版本，是现代大语言模型（LLaMA、Nemotron等）的标配。
    普通LayerNorm公式: output = (x - mean) / std * weight + bias
    RMSNorm公式:output = x / RMS(x) * weight
    其中 RMS(x) = sqrt(mean(x²))，比LayerNorm少了减均值这一步，更快。
    为什么要替换：Triton的RMSNorm实现在Blackwell GPU上有bug，用纯PyTorch绕过。
    
    参数：
        x      : 输入张量，形状通常是 [batch_size, seq_len, hidden_dim]
        weight : 可学习的缩放参数gamma，形状 [hidden_dim]
        bias   : 可选偏置（大多数RMSNorm没有bias）
        z      : 可选门控信号，用于SwiGLU结构
        eps    : 防止除以零的极小值，默认1e-5（即0.00001）
        upcast : 是否先转成float32再计算（提高数值稳定性）
    """
    dtype = x.dtype
    # 记住输入的原始数据类型（如bfloat16），最后要转回来

    if upcast:
        x = x.float()
    # 如果upcast=True，先把x转成float32（32位浮点）
    # 因为bfloat16精度低，计算归一化时容易有误差
    # 用float32计算完后再转回bfloat16

    variance = x.pow(2).mean(-1, keepdim=True)
    # x.pow(2)：每个元素平方 → x²
    # .mean(-1, ...)：对最后一维（hidden_dim）求平均 → mean(x²)
    # keepdim=True：保持维度不变（[batch, seq,1]），方便后面广播运算
    # 这就是RMS的平方（方差，但不减均值）

    x_normed = x * torch.rsqrt(variance + eps)
    # torch.rsqrt(y) = 1 / sqrt(y)（倒数平方根，比先sqrt再除法更高效）
    # variance + eps：防止分母为零
    # 整体：x / sqrt(mean(x²) + eps) = RMS归一化后的x

    out = x_normed * weight.float()
    # 乘以可学习的缩放权重weight（也转成float32保证精度）

    if bias is not None:
        out = out + bias.float()
    # 如果有偏置项，加上去

    if z is not None:
        out = out * F.silu(z.float())
    # SiLU激活函数：SiLU(x) = x * sigmoid(x)，也叫Swish
    # 如果有门控信号z，用SiLU处理后与out相乘
    # 这是SwiGLU结构，很多现代LLM用这个作为FFN激活函数

    return out.to(dtype)
    # 把float32结果转回原来的数据类型（如bfloat16）

# 遍历Python已加载的所有模块，把rmsnorm_fn替换成我们的纯Python版本
for name, mod in list(sys.modules.items()):
    # sys.modules：字典，记录所有已导入的模块 {模块名: 模块对象}
    # list()包裹：因为不能在遍历时修改字典，先转成列表快照
    if hasattr(mod, 'rmsnorm_fn'):
        # hasattr：检查模块是否有rmsnorm_fn这个属性/函数
        mod.rmsnorm_fn = _pure_rmsnorm_fn
        # 直接覆盖！把有bug的Triton实现换成我们的安全版本

print("✅ RMSNorm Triton修复完成")

# ---- 修复2：修复ptxas编译器路径 ----
# ptxas = Parallel Thread eXecution Assembler，NVIDIA GPU汇编编译器
# Triton需要ptxas来把GPU内核代码编译成可执行的机器码
src = "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/triton/backends/nvidia/bin/ptxas-blackwell"
# Kaggle预置的Blackwell架构专用ptxas编译器位置

dst = "/tmp/ptxas-blackwell"
# 把它复制到/tmp目录（临时目录，我们有完全读写权限）

if os.path.exists(src):
    # 只有源文件存在才进行修复
    shutil.copy2(src, dst)
    # shutil.copy2：复制文件，同时保留原文件的元数据（时间戳等）# 给复制过来的文件添加可执行权限
    os.chmod(
        dst,
        os.stat(dst).st_mode# 获取当前权限位| stat.S_IEXEC           # 添加：文件所有者可执行
        | stat.S_IXGRP           # 添加：同组用户可执行
        | stat.S_IXOTH           # 添加：其他用户可执行
    )
    # 为什么要这一步：复制后的文件默认可能没有执行权限，ptxas需要执行权限才能运行

    import triton.backends.nvidia as nv_backend
    # 导入Triton的NVIDIA后端模块

    src_bin = os.path.join(os.path.dirname(nv_backend.__file__), "bin")
    # os.path.dirname：取文件所在目录
    # nv_backend.__file__：这个模块文件在磁盘上的路径
    # 拼接"bin"：得到Triton NVIDIA后端的可执行文件目录

    dst_bin = "/tmp/triton_nvidia_bin"
    shutil.copytree(src_bin, dst_bin, dirs_exist_ok=True)
    # copytree：复制整个目录树（包括子目录）
    # dirs_exist_ok=True：目标目录已存在也不报错（Python3.8+特性）

    # 给复制后的所有文件添加可执行权限
    for f in os.listdir(dst_bin):
        fp = os.path.join(dst_bin, f)
        if os.path.isfile(fp):
            # 只处理文件，跳过子目录
            os.chmod(
                fp,
                os.stat(fp).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH
            )# 修改nv_backend的文件路径，让Triton从新位置加载资源
    nv_backend.__file__ = os.path.join(dst_bin, "..", "__init__.py")

    # 设置环境变量：告诉Triton去哪里找ptxas-blackwell编译器
    os.environ["TRITON_PTXAS_PATH"] = dst

    print("✅ Triton ptxas修复完成")
else:
    # 如果预置的ptxas文件不存在，给出警告
    print("⚠️ 未找到ptxas-blackwell，跳过Triton修复")
    print("   如果训练时出现Triton编译错误，请检查Kaggle环境配置")

✅ RMSNorm Triton修复完成
✅ Triton ptxas修复完成


### 📊 Training Config + Dataset

**Hyperparameters:**
- LoRA rank = 32
- Seq length = 2048 → supports reasoning chains
- Batch size = 1 + grad accumulation = 4
- LR = 2e-4, epochs = 2

**Data:**
- Loads Nemotron reasoning dataset
- Subsamples 600 rows (fast experimentation)
- Converts to HuggingFace Dataset

✅ Balanced for:
  - H100 VRAM usage
  - Reasoning fine-tuning

In [4]:
# === Hyperparameters ===
SUBSAMPLE_SIZE = 2000   # 原版600，增加到2000效果更好
LORA_RANK = 32
LORA_ALPHA     = 64# ← 补上这行！alpha = 2×rank是最佳实践
LORA_DROPOUT   = 0.05      # ← 补上这行！防止过拟合
MAX_SEQ_LEN = 2048      # Increased from 1024 to support reasoning chains
NUM_EPOCHS = 3          # 原版2，增加到3收敛更充分
BATCH_SIZE = 1
GRAD_ACCUM = 4
LR = 1e-4
LORA_ALPHA = 64# 新增，原版没有单独列出（默认32），改为64更好
OUTPUT_DIR = "/kaggle/working/adapter"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# === Data ===
MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")

# 自动查找CSV，防止文件名不匹配
import glob
_candidates = glob.glob("/kaggle/input/**/*.csv", recursive=True)
# 选最大的CSV（训练集通常最大）
_candidates = sorted(_candidates, key=lambda f: os.path.getsize(f), reverse=True)
if not _candidates:
    raise FileNotFoundError("未找到任何CSV！请在右侧 + 添加数据集")
DATA_PATH = _candidates[0]
print(f"使用数据文件: {DATA_PATH}")

train_df = pl.read_csv(DATA_PATH)
train_df = train_df.sample(n=min(SUBSAMPLE_SIZE, len(train_df)), seed=42)
hf_dataset = Dataset.from_pandas(train_df.to_pandas())
print(f"数据加载完成: {len(hf_dataset)} 条，列名: {hf_dataset.column_names}")

使用数据文件: /kaggle/input/datasets/kienngx/nemotron-30b-competition-trainingdata-cot-labels/final_Nemotron_training_data.csv
数据加载完成: 2000 条，列名: ['id', 'prompt', 'answer', 'generated_cot', 'label']


### 💬 Training Prompt Construction

- Converts dataset into chat format:
  - User: problem + instruction (`\\boxed{}`)
  - Assistant: full answer (reasoning + final)

- Uses `tokenizer.apply_chat_template`
  → ensures model-native formatting

- Fallback to manual template if needed

⚠️ Critical assumption:
- `answer` must contain reasoning BEFORE final result

✅ Aligns training with inference behavior

In [5]:
# ============================================================
# Cell 5: Tokenizer加载与训练文本构建
# Tokenizer：把文字转成模型能处理的数字序列
# 训练文本：把"题目+推理链+答案"拼成模型期望的对话格式
# ============================================================

print("📥 正在加载Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_PATH,
    trust_remote_code=True# trust_remote_code=True：允许执行模型自带的自定义代码
    # Nemotron模型有自己定制的tokenizer实现，需要这个权限
)

# ---- 设置padding token----
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token# pad_token（填充token）：当一批样本长度不同时，短的要用pad_token补齐
    # eos_token（End of Sequence）：句子结束符号
    # 如果模型没定义pad_token，就借用eos_token充当
    # 这对因果语言模型（GPT类）是标准做法

print(f"✅ Tokenizer加载完成")
print(f"   词表大小: {tokenizer.vocab_size}")
print(f"   pad_token: '{tokenizer.pad_token}'")
print(f"   eos_token: '{tokenizer.eos_token}'")

# ---- 构建训练文本的函数 ----
def build_training_text(example):
    """
    把数据集的一行转换成训练用的对话格式文本。
    数据集每行包含：
        - prompt: 题目（比如数学题、推理题）
        - answer: 正确答案（如"42" 或 "Paris"）
        - generated_cot: 思维链（Chain of Thought）也就是分步骤的推理过程
    
    训练目标：让模型学会：
        看到题目 → 先写出完整推理过程 → 最后给出\boxed{答案}
    
    为什么要CoT（思维链）？
        - 直接回答正确率低
        - 让模型一步步推理，准确率更高
        - 比赛评分要求答案放在\boxed{}里
    """

    prompt = example["prompt"]          # 题目文字
    answer = str(example["answer"])     # 转成字符串（防止数字类型问题）
    cot    = example["generated_cot"]   # 推理链文字

    # ---- 构建用户消息 ----
    user_msg = prompt + "\nPut your final answer inside\\boxed{}."
    #在题目后面加指令：把最终答案放在\boxed{}里
    # \\boxed{}：Python字符串里\\是转义，实际输出的是 \boxed{}
    # \boxed{} 是LaTeX数学公式命令，比赛评分系统从这里提取答案

    # ---- 构建助手回复（这就是模型要学的输出格式） ----
    assistant_msg = f"{cot}\n\n\\boxed{{{answer}}}"
    # f-string格式化：
    #   {cot}     → 插入推理链文字
    #   \n\n      → 两个空行（分隔推理和答案）
    #   \\boxed   → 输出字面量 \boxed（\\转义为\）
    #   {{{answer}}} → 外层{{ }}转义为{ }，内层{answer}插入变量值
    #   最终效果：\boxed{42}（如果answer是42）

    # ---- 格式化成对话格式 ----
    try:
        # 优先尝试用模型自带的对话模板
        messages = [
            {"role": "user","content": user_msg},
            {"role": "assistant", "content": assistant_msg},
        ]
        # 标准对话格式：role指定说话者，content是内容
        # user =用户提问，assistant = AI回答

        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,              # 只转成字符串，不转成数字token
            add_generation_prompt=False# 不在末尾加生成提示，因为回答已经写好了
        )
        # apply_chat_template：把对话列表格式化成模型期望的字符串
        # 不同模型有不同的格式，这个函数自动处理

    except Exception:
        # 如果模型没有对话模板，手动用ChatML格式
        # ChatML是OpenAI定义的通用对话格式，很多开源模型支持
        text = (
            f"<|im_start|>user\n{user_msg}<|im_end|>\n"
            f"<|im_start|>assistant\n{assistant_msg}<|im_end|>"
        )# <|im_start|>：消息开始标记
        # <|im_end|>：消息结束标记

    return {"text": text}
    # 返回字典，键必须是"text"，这是SFTTrainer期望的格式

# ---- 把函数应用到整个数据集 ----
print("⚙️ 正在构建训练文本...")
hf_dataset = hf_dataset.map(
    build_training_text,
    remove_columns=hf_dataset.column_names# remove_columns：删除原来的所有列（prompt, answer, cot等）
    # 只保留新生成的"text"列
)
print(f"✅ 训练文本构建完成，共 {len(hf_dataset)} 条")

# ---- 长度过滤（重要的优化步骤！） ----
print("✂️ 正在过滤超长样本...")

def filter_by_length(example):
    """
    过滤掉token数量超过MAX_SEQ_LEN的样本。
    超长样本在训练时会被截断，截断会破坏答案格式（\boxed{}可能被截掉），
    白白占用计算时间，不如直接过滤掉。
    """
    # 把文本转成token，return_length=True会直接返回长度
    tokens = tokenizer(
        example["text"],
        return_length=True,
        truncation=False# 不截断，我们要测真实长度
    )
    return tokens["length"][0] <= MAX_SEQ_LEN
    # 只保留长度不超过MAX_SEQ_LEN的样本

hf_dataset = hf_dataset.filter(filter_by_length)
print(f"✅ 长度过滤后剩余: {len(hf_dataset)} 条样本")

# ---- 展示一个完整样本，验证格式正确 ----
print("\n📋 训练文本示例（前300字符）:")
print(hf_dataset[0]["text"][:300])
print("...")

📥 正在加载Tokenizer...
✅ Tokenizer加载完成
   词表大小: 131072
   pad_token: '<|im_end|>'
   eos_token: '<|im_end|>'
⚙️ 正在构建训练文本...


Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

✅ 训练文本构建完成，共 2000 条
✂️ 正在过滤超长样本...


Filter:   0%|          | 0/2000 [00:00<?, ? examples/s]

✅ 长度过滤后剩余: 1958 条样本

📋 训练文本示例（前300字符）:
<|im_start|>system
<|im_end|>
<|im_start|>user
In Alice's Wonderland, a secret unit conversion is applied to measurements. For example:
10.3 m becomes 10.40
12.54 m becomes 12.66
5.85 m becomes 5.91
Now, convert the following measurement: 11.01 m
Put your final answer inside\boxed{}.<|im_end|>
<|im_
...


### 🧠 Model Loading & LoRA Fine-tuning

**Model:**
- Loads Nemotron-3 Nano 30B (bf16, GPU)
- Enables gradient checkpointing → saves VRAM

**Patching:**
- Disables fast path kernels (stability fix)

**LoRA:**
- Rank = 32, alpha = 32
- Targets all major projection layers
- Reduces trainable params significantly

**Training:**
- Uses `SFTTrainer` (TRL)
- Cosine LR scheduler + warmup
- No checkpoint saving (Kaggle constraint)

✅ Efficient fine-tuning of 30B on single H100

In [6]:
# ============================================================
# Cell 6: 模型加载与LoRA配置
# 加载30B基础模型，然后给它套上LoRA"插件"
# ============================================================

print("📥 正在加载30B基础模型（这需要几分钟...）")

# ---- 加载基础模型 ----
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,

    device_map={"": 0},
    # 把整个模型放到第0号GPU（即第一张显卡）
    # 如果有多张GPU，可以改成 device_map="auto" 自动分配
    # {"": 0} 的含义："": 代表所有层，0代表GPU0

    trust_remote_code=True,
    # 允许执行Nemotron模型自带的自定义代码
    # 必须设置，否则Nemotron模型无法正确加载

    torch_dtype=torch.bfloat16,
    # 用bfloat16精度加载模型权重
    # bfloat16 vs float32：内存减半，精度损失很小
    # bfloat16 vs float16：数值范围更大，不容易出现NaN（数值崩溃）
    # 30B参数 × 2字节(bfloat16) ≈ 60GB显存
)

print("✅ 模型加载完成")
print(f"   模型参数量: {sum(p.numel() for p in model.parameters()) / 1e9:.1f}B")

# ---- 开启梯度检查点 ----
model.gradient_checkpointing_enable()
#梯度检查点（Gradient Checkpointing）是一种省显存技术：
#
# 正常训练：前向传播时保存所有中间激活值（占大量显存）
#           反向传播时直接用保存的激活值计算梯度
#
# 开启后：只在关键"检查点"保存激活值，其余在反向传播时重新计算
#         显存减少约30-40%，但训练速度降低约15-20%
#
# 对30B模型来说这是必须的，否则显存不够用

# ---- 修复Nemotron的快速路径问题 ----
for name, mod in sys.modules.items():
    if "modeling_nemotron_h" in name:
        # 找到所有Nemotron模型相关的模块
        mod.is_fast_path_available = False
        # 禁用"快速路径"（fast path）
        # 快速路径是Triton优化的计算路径，在Kaggle Blackwell GPU上有兼容性问题
        # 禁用后使用普通PyTorch实现，稍慢但稳定
        print(f"✅ 已修补 {name}: is_fast_path_available = False")

# ---- 配置LoRA ----
print("\n⚙️ 正在配置LoRA...")

lora_config = LoraConfig(
    r=LORA_RANK,
    #LoRA的秩 = 32（比赛上限）
    # 秩越大，可训练参数越多，表达能力越强

    lora_alpha=LORA_ALPHA,
    # 缩放因子 = 64
    # 实际缩放比 = lora_alpha / r = 64 / 32 = 2.0
    # 经验规则：alpha =2 × r 效果通常好于 alpha = r

    target_modules=[
        # 明确指定对哪些层应用LoRA（比"all-linear"更精确）
        # 这些是Transformer模型中最重要的线性层：
        "q_proj",    # Query投影矩阵（注意力机制的查询向量）
        "k_proj",    # Key投影矩阵（注意力机制的键向量）
        "v_proj",    # Value投影矩阵（注意力机制的值向量）
        "o_proj",    # 注意力输出投影
        "gate_proj", # FFN（前馈网络）的门控层
        "up_proj",   # FFN的上投影层
        "down_proj", # FFN的下投影层
        # 这7个层覆盖了Transformer的核心计算部分
        # 比"all-linear"更有针对性，避免对embedding等无关层微调
    ],

    lora_dropout=LORA_DROPOUT,
    # dropout = 0.05，训练时随机关闭5%的连接
    # 防止过拟合，数据少时很有用

    bias="none",
    # 不对偏置项应用LoRA（通常不需要，节省参数）

    task_type=TaskType.CAUSAL_LM,
    # 任务类型：因果语言模型（GPT类模型的标准设置）
    # CAUSAL_LM = 每个位置只能看到它之前的token，不能"偷看"未来
)

# 把普通模型包装成带LoRA的PEFT模型
model = get_peft_model(model, lora_config)
#这一步会：
# 1. 冻结(freeze)原始模型的所有参数（不参与训练）
# 2. 在指定的层旁边添加LoRA的A、B小矩阵
# 3. 只有LoRA矩阵是可训练的

# 打印可训练参数量，验证LoRA配置正确
model.print_trainable_parameters()
# 输出示例：
# trainable params: 167,772,160|| all params: 30,167,772,160 || trainable%: 0.556%
# 说明只有不到1%的参数在训练，极大节省了计算量

# ---- 再次修复Triton编译器 ----
# （这是第二次修复，在模型加载后的Triton相关设置）
try:
    import triton.backends.nvidia.compiler as nv_compiler

    os.environ["TRITON_PTXAS_BLACKWELL_PATH"] = "/tmp/ptxas-blackwell"
    # 告诉Triton Blackwell GPU的ptxas编译器在哪里

    nv_compiler.get_ptxas_version = lambda arch: "12.0"
    # lambda arch: "12.0" 创建一个匿名函数：接受arch参数，总返回"12.0"
    # 强制ptxas版本为"12.0"，绕过Triton的版本兼容性检查
    # 在Kaggle Blackwell环境中，Triton可能误判版本号

    print("✅ Triton编译器版本修复完成")
except ImportError:
    print("⚠️ triton未找到，跳过编译器修复")

`torch_dtype` is deprecated! Use `dtype` instead!


📥 正在加载30B基础模型（这需要几分钟...）


Loading weights:   0%|          | 0/6243 [00:00<?, ?it/s]

✅ 模型加载完成
   模型参数量: 31.6B
✅ 已修补 transformers_modules._1.modeling_nemotron_h: is_fast_path_available = False

⚙️ 正在配置LoRA...


/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_tensor.py:122: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_in_graph
/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_tensor.py:195: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_in_graph
/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_scaling_utils.py:90: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_in_graph
/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_linear.py:60: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_

trainable params: 869,318,656 || all params: 32,447,256,000 || trainable%: 2.6792
✅ Triton编译器版本修复完成


### 💾 Save LoRA Adapter

- Saves:
  - `adapter_model.safetensors`
  - `adapter_config.json`
  - tokenizer config

- Prints file sizes for verification

✅ Only LoRA weights saved (NOT full model)
→ lightweight (~MBs instead of GBs)

In [ ]:
# ============================================================
# Cell 7: 训练配置与执行（修正版）
# ============================================================

import os
import gc
import torch
from transformers import TrainerCallback
from trl import SFTTrainer, SFTConfig

# ---- 自定义训练监控回调 ----
class TrainingMonitorCallback(TrainerCallback):
    """
    在训练过程中打印关键信息，便于观察loss走势和训练状态
    """

    def on_train_begin(self, args, state, control, **kwargs):
        print("\n" + "=" * 60)
        print("🚀 训练开始")
        print(f"   num_train_epochs: {args.num_train_epochs}")
        print(f"   learning_rate   : {args.learning_rate}")
        print(f"   batch_size/device: {args.per_device_train_batch_size}")
        print(f"   grad_accum_steps : {args.gradient_accumulation_steps}")
        print("=" * 60)

    def on_log(self, args, state, control, logs=None, **kwargs):
        """
        每次记录日志时触发（由 logging_steps 控制）
        """
        if not logs:
            return

        step = state.global_step
        loss = logs.get("loss", None)
        lr = logs.get("learning_rate", None)

        if loss is not None:
            lr_str = f"{lr:.2e}" if lr is not None else "N/A"
            print(f"  Step {step:4d} | Loss: {loss:.4f} | LR: {lr_str}")

    def on_epoch_end(self, args, state, control, **kwargs):
        """
        每个 epoch 结束触发
        """
        epoch = state.epoch if state.epoch is not None else -1

        # 安全获取最近一次loss
        last_loss = "N/A"
        for item in reversed(state.log_history):
            if "loss" in item:
                last_loss = f"{item['loss']:.4f}"
                break

        print(f"\n🎯 Epoch {epoch:.0f} 训练完成！当前Loss：{last_loss}")

        # 可选清理（防碎片）
        torch.cuda.empty_cache()
        gc.collect()
        print("   GPU显存已清理\n")

    def on_train_end(self, args, state, control, **kwargs):
        print("=" * 60)
        print("✅ 训练结束")
        print(f"   global_step: {state.global_step}")

        losses = [x["loss"] for x in state.log_history if "loss" in x]
        if len(losses) > 0:
            print(f"   最终loss: {losses[-1]:.4f}")
            print(f"   最低loss: {min(losses):.4f}")
        print("=" * 60)

# ---- 配置训练参数 ----
print("⚙️ 配置训练参数...")

training_args = SFTConfig(
    # 基本
    output_dir=OUTPUT_DIR,

    # batch
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,

    # epoch
    num_train_epochs=NUM_EPOCHS,

    # lr
    learning_rate=LR,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,

    # 优化与精度
    bf16=True,
    optim="adamw_torch",
    max_grad_norm=1.0,

    # 显存优化
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},

    # 文本与长度
    max_length=MAX_SEQ_LEN,
    dataset_text_field="text",
    packing=True,  # 继续保留

    # 日志与保存
    logging_steps=10,          # 建议先10，loss看得更清楚
    save_strategy="no",
    report_to="none",
)

print("✅ 训练配置完成")
print(f"\n📊 数据量: {len(hf_dataset)}")
print(f"   训练轮数: {NUM_EPOCHS}")
print(f"   max_seq_len: {MAX_SEQ_LEN}")
print(f"   学习率: {LR}")
print(f"   等效batch大小: {BATCH_SIZE * GRAD_ACCUM}")

# ---- 创建训练器 ----
trainer = SFTTrainer(
    model=model,
    train_dataset=hf_dataset,
    processing_class=tokenizer,
    args=training_args,
    callbacks=[TrainingMonitorCallback()],
)

# ---- 开始训练 ----
print("\n🚀 开始训练！")
trainer.train()   # 只调用一次
print("\n✅ 训练完成！")

# ---- 保存LoRA适配器 ----
print(f"💾 正在保存LoRA适配器到: {OUTPUT_DIR}")
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

# ---- 打印输出文件 ----
print("\n📁 已保存文件:")
for f in sorted(os.listdir(OUTPUT_DIR)):
    p = os.path.join(OUTPUT_DIR, f)
    size_bytes = os.path.getsize(p)
    size_str = f"{size_bytes/1024/1024:.1f} MB" if size_bytes > 1024*1024 else f"{size_bytes/1024:.1f} KB"
    print(f"   {f:<40} {size_str}")

# ---- 收尾清理 ----
del trainer
torch.cuda.empty_cache()
gc.collect()
print("\n✅ GPU显存已清理，流程结束")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


⚙️ 配置训练参数...
✅ 训练配置完成

📊 数据量: 1958
   训练轮数: 3
   max_seq_len: 2048
   学习率: 0.0001
   等效batch大小: 4


Adding EOS to train dataset:   0%|          | 0/1958 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1958 [00:00<?, ? examples/s]

Packing train dataset:   0%|          | 0/1958 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 11, 'pad_token_id': 11}.



🚀 开始训练！

🚀 训练开始
   num_train_epochs: 3
   learning_rate   : 0.0001
   batch_size/device: 1
   grad_accum_steps : 4


### 📦 Create Submission ZIP

- Compresses adapter files into `submission.zip`
- Verifies required files exist:
  - `adapter_config.json`
  - `adapter_model.safetensors`

- Raises error if missing (prevents submission failure)

✅ Ready for Kaggle submission
⚠️ Missing files = instant evaluation failure

In [ ]:
# ============================================================
# Cell 8: 打包提交文件
# 把LoRA适配器文件打包成submission.zip
# 比赛要求提交的zip文件必须包含：
#   - adapter_config.json（LoRA配置）
#   - adapter_model.safetensors（LoRA权重）
#============================================================

import zipfile
import os

#---- 定义路径 ----
OUTPUT_DIR = "/kaggle/working/adapter"# LoRA文件所在目录
zip_path   = "/kaggle/working/submission.zip"  # 输出的zip文件路径

print(f"📦 正在打包 {OUTPUT_DIR} 中的文件...")

# ---- 创建zip文件 ----
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    # zipfile.ZipFile参数说明：
    #   zip_path：创建的zip文件路径
    #   'w'：写模式（write），如果文件已存在会覆盖
    #   zipfile.ZIP_DEFLATED：使用DEFLATE压缩算法（比STORED压缩率更高）

    for fname in os.listdir(OUTPUT_DIR):
        # os.listdir：列出目录下所有文件/文件夹名
        fpath = os.path.join(OUTPUT_DIR, fname)
        #拼接成完整路径

        if os.path.isfile(fpath):
            # 只处理文件，跳过子文件夹
            zf.write(fpath, fname)
            # zf.write参数：
            #   fpath：磁盘上文件的完整路径（从哪里读）
            #   fname：在zip内的存储路径（只用文件名，不带目录）
            # 效果：zip内部结构是扁平的，直接是文件名，没有子目录
            size_kb = os.path.getsize(fpath) / 1024
            print(f"✅ 已添加: {fname} ({size_kb:.1f} KB)")

# ---- 打印zip文件总大小 ----
zip_size_mb = os.path.getsize(zip_path) / 1024 / 1024
print(f"\n📦 zip文件创建成功: {zip_path}")
print(f"   文件大小: {zip_size_mb:.1f} MB")

# ---- 验证zip文件内容 ----
print("\n🔍 正在验证zip文件内容...")

with zipfile.ZipFile(zip_path, 'r') as zf:
    # 'r'：读模式（read）
    zip_contents = zf.namelist()
    # namelist()：返回zip内所有文件名的列表
    print(f"  zip内文件列表: {zip_contents}")

# ---- 关键文件检查 ----
required_files = ["adapter_config.json", "adapter_model.safetensors"]
# 比赛评分系统需要这两个文件才能加载LoRA适配器
# adapter_config.json：描述LoRA的配置（rank、alpha等），评分系统据此初始化LoRA结构
# adapter_model.safetensors：LoRA的实际权重，.safetensors格式比.bin更安全

all_good = True
for required_file in required_files:
    if required_file not in zip_contents:
        print(f"❌ 缺少必须文件: {required_file}")
        all_good = False
    else:
        print(f"  ✅ 找到必须文件: {required_file}")

if not all_good:
    raise AssertionError(
        "❌ zip文件缺少关键文件！提交到比赛会失败！\n"
        "请检查训练是否正常完成，OUTPUT_DIR中是否有上述文件。"
    )

# ---- 最终确认 ----
print("\n" + "="*50)
print("✅ submission.zip 已准备好，可以提交到比赛！")
print("="*50)
print(f"\n📋 提交检查清单:")
print(f"✅ adapter_config.json   - LoRA配置文件")
print(f"  ✅ adapter_model.safetensors - LoRA权重文件")
print(f"  ✅ zip大小: {zip_size_mb:.1f} MB")
print(f"\n🎯 下一步：在Kaggle competition页面点击 'Submit' 上传 submission.zip")